# 🏁 F1 Circuit Predictor - Exploratory Data Analysis (EDA)

This notebook contains the Exploratory Data Analysis (EDA) for our F1 Race Outcome Predictor. The purpose of this analysis is to explore the correlations between starting grid positions, constructor strength, weather conditions, and the final race outcome. This analysis will justify the feature engineering decisions made in our modeling pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline
sns.set_theme(style="darkgrid")

## 1. Load Processed Datasets

In [ ]:
processed_dir = "../data/processed"
results_path = os.path.join(processed_dir, "all_results.csv")
weather_path = os.path.join(processed_dir, "all_weather.csv")

if os.path.exists(results_path):
    df_results = pd.read_csv(results_path)
    print(f"Results loaded. Shape: {df_results.shape}")
    display(df_results.head(3))
else:
    print("⚠️ Results dataset not found. Generating a mock dataset for demonstration.")
    # Fallback mock data for EDA visualization
    np.random.seed(42)
    mock_size = 500
    grid = np.random.randint(1, 21, mock_size)
    # Finish position correlated with grid, but with random variance
    pos = np.clip(grid + np.random.normal(0, 3, mock_size).astype(int), 1, 20)
    df_results = pd.DataFrame({
        'Year': np.random.choice([2021, 2022, 2023, 2024], mock_size),
        'GP': np.random.choice(['Monaco GP', 'British GP', 'Italian GP', 'Abu Dhabi GP'], mock_size),
        'Session': ['R'] * mock_size,
        'Abbreviation': np.random.choice(['VER', 'HAM', 'LEC', 'NOR', 'ALO'], mock_size),
        'TeamName': np.random.choice(['Red Bull Racing', 'Mercedes', 'Ferrari', 'McLaren', 'Aston Martin'], mock_size),
        'GridPosition': grid,
        'Position': pos,
        'Points': np.random.choice([25, 18, 15, 12, 10, 8, 6, 4, 2, 1, 0], mock_size),
        'Status': np.random.choice(['Finished', 'Finished', 'Finished', 'Accident', 'Engine'], mock_size)
    })
    print(f"Mock results shape: {df_results.shape}")

## 2. Correlation Analysis: Grid Position vs. Finish Position

In F1, starting grid position is the single most predictive feature of the final race result. Let's quantify this relationship.

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(data=df_results[df_results['Session'] == 'R'], x='GridPosition', y='Position', 
            scatter_kws={'alpha':0.4, 'color':'#FF1801'}, line_kws={'color':'black', 'linewidth':2})
plt.title('Impact of Starting Grid Position on Final Race Finish', fontsize=14)
plt.xlabel('Starting Grid Position', fontsize=12)
plt.ylabel('Final Race Position', fontsize=12)
plt.xticks(range(1, 21))
plt.yticks(range(1, 21))
plt.show()

corr = df_results[df_results['Session'] == 'R'][['GridPosition', 'Position']].corr().iloc[0, 1]
print(f"Pearson Correlation Coefficient: {corr:.3f}")

## 3. Team and Driver Distribution

Different teams have different average performance metrics across seasons. Let's check constructor strength distribution.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_results[df_results['Session'] == 'R'], x='TeamName', y='Position', palette='Reds')
plt.title('Distribution of Finishing Positions by Team / Constructor', fontsize=14)
plt.xlabel('Constructor', fontsize=12)
plt.ylabel('Finishing Position', fontsize=12)
plt.xticks(rotation=45)
plt.show()

## 4. DNF (Did Not Finish) and Attrition Rates

DNFs represent reliability and driver errors. We can look at how DNF status relates to constructor and track status.

In [ ]:
df_results['Is_DNF'] = ~df_results['Status'].astype(str).str.contains('Finished|Lap', case=False, na=False)
df_results['Is_DNF'] = df_results['Is_DNF'].astype(int)

dnf_by_team = df_results[df_results['Session'] == 'R'].groupby('TeamName')['Is_DNF'].mean().reset_index().sort_values(by='Is_DNF', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=dnf_by_team, x='Is_DNF', y='TeamName', color='#FF1801')
plt.title('DNF Rate by Constructor', fontsize=14)
plt.xlabel('Average DNF Rate (Higher = Less Reliable)', fontsize=12)
plt.ylabel('Constructor', fontsize=12)
plt.show()

## 5. Conclusion and Feature Selection

Key takeaways from the EDA:
1. **Grid Position** shares a strong positive linear correlation with finishing position. This is the cornerstone of F1 prediction.
2. **Constructor/Team** distributions show significant variance, justifying the inclusion of `Team_Avg_Position_Season`.
3. **Driver Rolling Average** captures season-specific driver form and driver reliability (`Driver_Avg_Position_Season`, `Driver_DNF_Rate_Season`).